# Positional Encoding — Sinusoidal vs Learned

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

Self-attention is permutation-invariant: shuffle the tokens and the output shuffles too. To give the model a sense of order we *add* a position vector to each token embedding. Sinusoidal encodings have nice properties (extrapolation, relative shifts), but learned positions can be more flexible inside the training range.


## Mathematical Formulation

Sinusoidal:

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

For any fixed offset $k$, $PE_{pos+k}$ is a linear function of $PE_{pos}$, so the model can attend by relative position.


## Implementation


In [ ]:
import math, torch
import torch.nn as nn
import matplotlib.pyplot as plt


In [ ]:
def sinusoidal(L, d):
    pe = torch.zeros(L, d)
    pos = torch.arange(L).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

class LearnedPE(nn.Module):
    def __init__(self, max_len, d):
        super().__init__()
        self.pe = nn.Embedding(max_len, d)
    def forward(self, x):
        idx = torch.arange(x.size(1), device=x.device)
        return x + self.pe(idx)


## Experiment


In [ ]:
pe = sinusoidal(L=128, d=64)
plt.figure(figsize=(8, 4))
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')
plt.colorbar(label='value'); plt.xlabel('dim'); plt.ylabel('position')
plt.title('Sinusoidal positional encoding'); plt.show()


In [ ]:
# Check the relative-shift property: PE[pos+k] should be a linear function of PE[pos]
diff = torch.cdist(pe[:1], pe).squeeze()
plt.plot(diff.numpy())
plt.xlabel('position'); plt.ylabel('distance to PE[0]')
plt.title('Distance grows smoothly — supports relative reasoning'); plt.show()


## Discussion

- Sinusoidal: zero parameters, extrapolates to lengths longer than seen in training, encodes *relative* shifts cleanly.
- Learned: a few extra params per position, can capture peculiarities of a finite training set, but cannot extrapolate past the trained range.
- Modern alternatives include **rotary** (RoPE) and **ALiBi**, which inject position information directly inside attention rather than at the embedding level.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
